In [1]:
!pip install ultralytics


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os

img_path = "animal_dataset/images/train"
lbl_path = "animal_dataset/labels/train"

images = [f.replace(".jpg", "") for f in os.listdir(img_path) if f.endswith(".jpg")]
labels = [f.replace(".txt", "") for f in os.listdir(lbl_path) if f.endswith(".txt")]

missing_labels = []
missing_images = []

for img in images:
    if img not in labels:
        missing_labels.append(img)

for lbl in labels:
    if lbl not in images:
        missing_images.append(lbl)

print("Images without labels:", len(missing_labels))
print("Labels without images:", len(missing_images))


Images without labels: 0
Labels without images: 0


In [3]:
import os

def convert_to_single_class(label_folder):
    for file in os.listdir(label_folder):
        if file.endswith(".txt"):
            path = os.path.join(label_folder, file)

            with open(path, "r") as f:
                lines = f.readlines()

            new_lines = []
            for line in lines:
                parts = line.strip().split()
                parts[0] = "0"  # force to single class
                new_lines.append(" ".join(parts) + "\n")

            with open(path, "w") as f:
                f.writelines(new_lines)

convert_to_single_class("animal_dataset/labels/train")
convert_to_single_class("animal_dataset/labels/val")

print("All labels converted to class 0 (animal)")


All labels converted to class 0 (animal)


In [4]:
from collections import Counter
import os

counter = Counter()

for folder in ["animal_dataset/labels/train", "animal_dataset/labels/val"]:
    for file in os.listdir(folder):
        if file.endswith(".txt"):
            with open(os.path.join(folder, file)) as f:
                for line in f:
                    class_id = line.split()[0]
                    counter[class_id] += 1

print(counter)


Counter({'0': 8004})


In [9]:
import os
import cv2
import random

img_dir = "animal_dataset/images/train"
lbl_dir = "animal_dataset/labels/train"

images = [f for f in os.listdir(img_dir) if f.endswith(".jpg")]

sample_images = random.sample(images, 5)  # show 5 random images

for img_name in sample_images:
    img_path = os.path.join(img_dir, img_name)
    lbl_path = os.path.join(lbl_dir, img_name.replace(".jpg", ".txt"))

    image = cv2.imread(img_path)
    h, w, _ = image.shape

    if os.path.exists(lbl_path):
        with open(lbl_path, "r") as f:
            lines = f.readlines()

        for line in lines:
            parts = line.strip().split()
            x_center, y_center, width, height = map(float, parts[1:])

            # Convert YOLO format to pixel coords
            x1 = int((x_center - width/2) * w)
            y1 = int((y_center - height/2) * h)
            x2 = int((x_center + width/2) * w)
            y2 = int((y_center + height/2) * h)

            cv2.rectangle(image, (x1, y1), (x2, y2), (0,255,0), 2)
            cv2.putText(image, "ANIMAL", (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

    cv2.imshow("Check Annotation", image)
    cv2.waitKey(0)

cv2.destroyAllWindows()


In [11]:
from collections import Counter
import os

counter = Counter()
for folder in ["animal_dataset/labels/train", "animal_dataset/labels/val"]:
    for f in os.listdir(folder):
        if f.endswith(".txt"):
            for line in open(os.path.join(folder, f)):
                counter[line.split()[0]] += 1

print(counter)


Counter({'0': 8004})


In [2]:
from ultralytics import YOLO

model = YOLO("yolov8n.yaml")  # from scratch 

model.train(
    data="animal_dataset/data.yaml",
    epochs=40,          
    imgsz=640,          # Keep high resolution for accuracy
    batch=6,            # Safer for CPU
    optimizer="AdamW",  # Better convergence
    patience=15,        # Stop if no improvement
    cos_lr=True,        # Better learning rate scheduling
    augment=True        # Improve generalization
)


New https://pypi.org/project/ultralytics/8.4.14 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.204  Python-3.13.2 torch-2.8.0+cpu CPU (13th Gen Intel Core i5-13500H)
engine\trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=6, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=animal_dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.yaml, momentum=0.937, mosaic=1.0, multi_scale=False, name=train5, nbs=64, nms=False, opset=None, o

KeyboardInterrupt: 

In [3]:
from ultralytics import YOLO

model = YOLO("runs/detect/train2/weights/last.pt")

model.train(
    resume=True
)


New https://pypi.org/project/ultralytics/8.4.14 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.204  Python-3.13.2 torch-2.8.0+cpu CPU (13th Gen Intel Core i5-13500H)
engine\trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=6, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=animal_dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs\detect\train2\weights\last.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train2, nbs=64, nm

KeyboardInterrupt: 

In [1]:
from ultralytics import YOLO

model = YOLO("runs/detect/train2/weights/last.pt")

model.train(
    resume=True
)


New https://pypi.org/project/ultralytics/8.4.14 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.204  Python-3.13.2 torch-2.8.0+cpu CPU (13th Gen Intel Core i5-13500H)
engine\trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=6, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=animal_dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs\detect\train2\weights\last.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train2, nbs=64, nm

KeyboardInterrupt: 

In [1]:
from ultralytics import YOLO

model = YOLO("runs/detect/train2/weights/last.pt")

model.train(resume=True)



New https://pypi.org/project/ultralytics/8.4.14 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.204  Python-3.13.2 torch-2.8.0+cpu CPU (13th Gen Intel Core i5-13500H)
engine\trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=6, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=animal_dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs\detect\train2\weights\last.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train2, nbs=64, nm

KeyboardInterrupt: 

In [1]:
from ultralytics import YOLO

model = YOLO("runs/detect/train2/weights/last.pt")

model.train(resume=True)



New https://pypi.org/project/ultralytics/8.4.14 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.204  Python-3.13.2 torch-2.8.0+cpu CPU (13th Gen Intel Core i5-13500H)
engine\trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=6, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=animal_dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs\detect\train2\weights\last.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train2, nbs=64, nm

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000002BEFF018B90>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [15]:
from ultralytics import YOLO

model = YOLO("runs/detect/train2/weights/best.pt")
print(model.names)

{0: 'animal'}


Normal Detection

In [ ]:
from ultralytics import YOLO

# Load model before overfitting correction
model = YOLO("runs/detect/train2/weights/last.pt")

# Gentle rollback training
model.train(
    data="animal_dataset/data.yaml",
    epochs=5,                 # small correction only
    optimizer="AdamW",        # force optimizer
    lr0=0.0002,               # VERY LOW learning rate
    augment=False,            # stop heavy augmentation
    patience=0,
    name="rollback_model"     # creates clean folder name
)

New https://pypi.org/project/ultralytics/8.4.14 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.204  Python-3.13.2 torch-2.8.0+cpu CPU (13th Gen Intel Core i5-13500H)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=animal_dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs/detect/train2/weights/last.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=rollback_model

In [1]:
from ultralytics import YOLO

# Load latest rollback checkpoint
model = YOLO("runs/detect/rollback_model/weights/last.pt")

# Resume training automatically
model.train(resume=True)

New https://pypi.org/project/ultralytics/8.4.14 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.204  Python-3.13.2 torch-2.8.0+cpu CPU (13th Gen Intel Core i5-13500H)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=animal_dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs\detect\rollback_model\weights\last.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=rollba

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000024D02BE42B0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [ ]:
Normal Detection

In [1]:
from ultralytics import YOLO
import cv2

# ===============================
# LOAD MODEL
# ===============================
model = YOLO("runs/detect/rollback_model/weights/best.pt")

cap = cv2.VideoCapture("test_media/video_20.mp4")

CONF_THRESHOLD = 0.40


# ===============================
#  ANIMAL
# ===============================
def is_real_animal(x1, y1, x2, y2, frame):

    H, W, _ = frame.shape

    w = x2 - x1
    h = y2 - y1

    area_ratio = (w * h) / (W * H)
    cy = (y1 + y2) / 2
    aspect = w / h

    # Large valid animal near road
    if area_ratio > 0.05 and cy > H * 0.4:
        return True

    if area_ratio < 0.004:
        return False

    if y1 < H * 0.30:
        return False

    if aspect < 0.45:
        return False

    if aspect > 3.8:
        return False

    return True


# ===============================
# DETECTION LOOP
# ===============================
while cap.isOpened():

    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, conf=CONF_THRESHOLD, verbose=False)

    for r in results:

        if r.boxes is None:
            continue

        for box in r.boxes:

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])

            # ✅ FILTER PERSON / SHELTER
            if not is_real_animal(x1, y1, x2, y2, frame):
                continue

            # DRAW ONLY VALID ANIMAL
            cv2.rectangle(frame,
                          (x1, y1),
                          (x2, y2),
                          (0, 255, 0),
                          3)

            cv2.putText(frame,
                        f"Animal {conf:.2f}",
                        (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.7,
                        (0, 255, 0),
                        2)

    cv2.imshow("Animal Detection Only", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
VIDEOS(PERFECT CODE)

In [8]:
from ultralytics import YOLO
import cv2
import winsound
import time
from collections import defaultdict

# ===============================
# LOAD MODEL
# ===============================
model = YOLO("runs/detect/rollback_model/weights/best.pt")

cap = cv2.VideoCapture("test_media/video_6.mp4")

ALARM_PATH = "alarm.wav"

memory = defaultdict(int)

CONF_THRESHOLD = 0.40
MIN_STABLE_FRAMES = 3
MAX_MEMORY = 6

alarm_playing = False
prev_area = None
prev_time = time.time()


# ===============================
# FINAL TRUE ANIMAL FILTER
# ===============================
def is_real_animal(x1, y1, x2, y2, frame):

    H, W, _ = frame.shape

    w = x2 - x1
    h = y2 - y1

    area_ratio = (w * h) / (W * H)
    cy = (y1 + y2) / 2
    aspect = w / h

    # ✅ CLOSE ANIMAL OVERRIDE
    if area_ratio > 0.05 and cy > H * 0.4:
        return True

    # ✅ FAR ANIMAL SUPPORT
    if area_ratio < 0.004:
        return False

    # ✅ REMOVE SKY OBJECTS
    if y1 < H * 0.30:
        return False

    # ✅ REMOVE HUMANS (vertical shape)
    if aspect < 0.45:
        return False

    # ✅ REMOVE EXTREME WIDE OBJECTS
    if aspect > 3.8:
        return False

    return True


# ===============================
# MAIN LOOP
# ===============================
while cap.isOpened():

    ret, frame = cap.read()
    if not ret:
        break

    H, W, _ = frame.shape

    results = model(frame, conf=CONF_THRESHOLD, verbose=False)

    verified_boxes = []
    high_risk_flag = False

    for r in results:

        if r.boxes is None:
            continue

        for box in r.boxes:

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])

            # ---------- FILTER ----------
            if not is_real_animal(x1, y1, x2, y2, frame):
                continue

            w_box = x2 - x1
            h_box = y2 - y1
            ratio = (w_box * h_box) / (W * H)

            # ---------- TEMPORAL STABILITY ----------
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2
            key = (cx // 80, cy // 80)

            memory[key] = min(memory[key] + 1, MAX_MEMORY)

            if ratio < 0.05:
                if memory[key] < MIN_STABLE_FRAMES:
                    continue

            verified_boxes.append((x1, y1, x2, y2, conf))

    # ===============================
    # COLLISION ANALYSIS
    # ===============================
    for x1, y1, x2, y2, conf in verified_boxes:

        box_area = (x2 - x1) * (y2 - y1)
        frame_area = W * H
        ratio = box_area / frame_area

        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2

        # ---------- RISK ----------
        if ratio > 0.06 and cy > H * 0.35:
            risk = "HIGH"
            color = (0, 0, 255)
            high_risk_flag = True
        elif ratio > 0.025:
            risk = "MEDIUM"
            color = (0, 255, 255)
        else:
            risk = "LOW"
            color = (0, 255, 0)

        # ---------- DISTANCE ----------
        if ratio > 0.12:
            distance = "VERY CLOSE"
        elif ratio > 0.06:
            distance = "CLOSE"
        elif ratio > 0.02:
            distance = "FAR"
        else:
            distance = "VERY FAR"

        # ---------- DIRECTION ----------
        if cx < W * 0.33:
            direction = "LEFT"
        elif cx > W * 0.66:
            direction = "RIGHT"
        else:
            direction = "CENTER"

        # ---------- TTC ----------
        curr_time = time.time()
        ttc_text = "N/A"

        if prev_area is not None:
            growth = box_area - prev_area
            dt = curr_time - prev_time

            if growth > 0 and dt > 0:
                ttc = frame_area / growth

                if ttc < 5:
                    ttc_text = "IMMINENT"
                elif ttc < 10:
                    ttc_text = "SOON"
                else:
                    ttc_text = "SAFE"

        prev_area = box_area
        prev_time = curr_time

        # ---------- DRAW ----------
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)

        cv2.putText(frame,
                    f"Animal {conf:.2f} | {risk}",
                    (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    color,
                    2)

        cv2.putText(frame,
                    f"{distance} | {direction} | TTC:{ttc_text}",
                    (x1, y2 + 20),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    color,
                    2)

    # ===============================
    # ALERT SYSTEM
    # ===============================
    if high_risk_flag:

        cv2.putText(frame,
                    "ANIMAL AHEAD - SLOW DOWN",
                    (40, 60),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.2,
                    (0, 0, 255),
                    3)

        if not alarm_playing:
            winsound.PlaySound(ALARM_PATH,
                               winsound.SND_ASYNC)
            alarm_playing = True
    else:
        if alarm_playing:
            winsound.PlaySound(None,
                               winsound.SND_PURGE)
            alarm_playing = False

    cv2.imshow("Final Animal Collision Detection", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
Final Animal–Vehicle Collision Detection Code

In [1]:
from ultralytics import YOLO
import cv2
import winsound
import time
from collections import defaultdict

# ===============================
# LOAD MODEL
# ===============================
model = YOLO("runs/detect/rollback_model/weights/best.pt")

cap = cv2.VideoCapture("test_media/video_6.mp4")

ALARM_PATH = "alarm.wav"

memory = defaultdict(int)

CONF_THRESHOLD = 0.40
MIN_STABLE_FRAMES = 3
MAX_MEMORY = 6

alarm_playing = False
prev_area = None
prev_time = time.time()


# ===============================
# FINAL TRUE ANIMAL
# ===============================
def is_real_animal(x1, y1, x2, y2, frame):

    H, W, _ = frame.shape

    w = x2 - x1
    h = y2 - y1

    area_ratio = (w * h) / (W * H)
    cy = (y1 + y2) / 2
    aspect = w / h

    if area_ratio > 0.05 and cy > H * 0.4:
        return True

    if area_ratio < 0.004:
        return False

    if y1 < H * 0.30:
        return False

    if aspect < 0.45:
        return False

    if aspect > 3.8:
        return False

    return True


# ===============================
# MAIN LOOP
# ===============================
while cap.isOpened():

    ret, frame = cap.read()
    if not ret:
        break

    H, W, _ = frame.shape

    results = model(frame, conf=CONF_THRESHOLD, verbose=False)

    verified_boxes = []
    high_risk_flag = False

    for r in results:

        if r.boxes is None:
            continue

        for box in r.boxes:

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])

            if not is_real_animal(x1, y1, x2, y2, frame):
                continue

            w_box = x2 - x1
            h_box = y2 - y1
            ratio = (w_box * h_box) / (W * H)

            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2
            key = (cx // 80, cy // 80)

            memory[key] = min(memory[key] + 1, MAX_MEMORY)

            if ratio < 0.05:
                if memory[key] < MIN_STABLE_FRAMES:
                    continue

            verified_boxes.append((x1, y1, x2, y2, conf))

    # ===============================
    # COLLISION ANALYSIS
    # ===============================
    for x1, y1, x2, y2, conf in verified_boxes:

        box_area = (x2 - x1) * (y2 - y1)
        frame_area = W * H
        ratio = box_area / frame_area

        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2

        # ✅ ALERT ZONE
        if ratio > 0.09 and cy > H * 0.45 and (W*0.35 < cx < W*0.65):
            risk = "HIGH"
            color = (0, 0, 255)
            high_risk_flag = True

        elif ratio > 0.03:
            risk = "MEDIUM"
            color = (0, 255, 255)

        else:
            risk = "LOW"
            color = (0, 255, 0)

        # ---------- DISTANCE ----------
        if ratio > 0.12:
            distance = "VERY CLOSE"
        elif ratio > 0.06:
            distance = "CLOSE"
        elif ratio > 0.02:
            distance = "FAR"
        else:
            distance = "VERY FAR"

        # ---------- DIRECTION ----------
        if cx < W * 0.33:
            direction = "LEFT"
        elif cx > W * 0.66:
            direction = "RIGHT"
        else:
            direction = "CENTER"

        # ---------- TTC ----------
        curr_time = time.time()
        ttc_text = "N/A"

        if prev_area is not None:
            growth = box_area - prev_area
            dt = curr_time - prev_time

            if growth > 0 and dt > 0:
                ttc = frame_area / growth

                if ttc < 5:
                    ttc_text = "IMMINENT"
                elif ttc < 10:
                    ttc_text = "SOON"
                else:
                    ttc_text = "SAFE"

        prev_area = box_area
        prev_time = curr_time

        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)

        cv2.putText(frame,
                    f"Animal {conf:.2f} | {risk}",
                    (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    color,
                    2)

        cv2.putText(frame,
                    f"{distance} | {direction} | TTC:{ttc_text}",
                    (x1, y2 + 20),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    color,
                    2)

    # ===============================
    # ALERT SYSTEM
    # ===============================
    if high_risk_flag:

        cv2.putText(frame,
                    "ANIMAL AHEAD - SLOW DOWN",
                    (40, 60),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.2,
                    (0, 0, 255),
                    3)

        if not alarm_playing:
            winsound.PlaySound(ALARM_PATH,
                               winsound.SND_ASYNC)
            alarm_playing = True
    else:
        if alarm_playing:
            winsound.PlaySound(None,
                               winsound.SND_PURGE)
            alarm_playing = False

    cv2.imshow("Animal Collision Detection", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()